[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/066.%20The%20Workforce%20Scheduling%20for%20Peak_Off-Peak%20Problem/P66-Tier-8_executed.ipynb)

---


# 66. The Workforce Scheduling for Peak/Off-Peak Problem
## Tier 8: A Value-Aligned and Ethical Scheduling Framework

### Purpose and scope

This notebook demonstrates a workforce-scheduling framework that scores randomly generated schedules using stakeholder utilities and penalties for ethical-constraint violations. It aims to make the trade-offs among employee workload balance, customer coverage, staffing cost, community employment, and selected ethical principles explicit.

The input is a demand value for each time period and a pool of employees. The output is the highest-scoring schedule among a fixed number of randomly generated candidates, together with stakeholder utility values, ethical-compliance checks, and diagnostic plots. The selected schedule is a best candidate from this sampling procedure; it is not a proof of a globally optimal schedule.

### Modeling assumptions

- Each schedule entry is binary: an employee is either assigned to a period or not assigned.
- Each period represents one hour, so an employee's row sum is their number of assigned work hours in the planning horizon.
- Candidate generation assigns exactly the requested demand in each period. The supplied demand must therefore be nonnegative, integer-valued, and no greater than the employee-pool size.
- Stakeholder utilities are combined using fixed weights. Ethical constraints are checked separately and violations incur fixed penalties; they are not enforced as hard constraints.
- The model does not represent shift lengths, rest periods, employee qualifications or preferences, wages, travel, or uncertainty in demand. Its sustainability measure is a staffing-count threshold, not a direct estimate of environmental impact.

### Stakeholders and principles

- **Employees:** prefer balanced workloads.
- **Customers:** prefer staffing in every period.
- **Shareholders:** prefer fewer assigned staff, as a simple proxy for cost efficiency.
- **Community:** prefers employment opportunities to be spread across the employee pool.
- **Fairness:** limits the ratio between the largest and smallest positive workloads.
- **Equity:** requires a specified share of employees to receive work.
- **Sustainability:** limits the total number of assignments as a simple proxy for resource use.

These definitions are illustrative model choices. In particular, a utility score or proxy metric is not, by itself, evidence that a real-world ethical outcome has been achieved.

## 1. Mathematical model and solution procedure

A candidate schedule can be represented by a binary matrix. Rows correspond to employees and columns to time periods. A value of one means that the employee is assigned to that period.

Let $i \in \{1,\ldots,n\}$ index employees and $j \in \{1,\ldots,T\}$ index periods. Let $d_j$ be demand in period $j$, and let $x_{ij} \in \{0,1\}$ indicate whether employee $i$ is assigned in period $j$. Candidate generation in this notebook imposes

$$
\sum_{i=1}^{n} x_{ij} = d_j \qquad \text{for each period } j.
$$

This equality is attainable when $0 \leq d_j \leq n$. The code constructs each candidate by sampling $d_j$ distinct employees for period $j$, so the equality holds by construction for the supplied data.

For stakeholder $s$, let $u_s(X)$ be the utility of schedule $X$, and let $w_s$ be that stakeholder's weight. For ethical condition $r$, let $g_r(X)$ equal one when the condition is satisfied and zero otherwise, and let $\lambda_r$ be its penalty weight. The score evaluated by the scheduler is

$$
F(X) = \sum_s w_s u_s(X) - \sum_r \lambda_r\bigl(1-g_r(X)\bigr).
$$

The method generates $K=100$ candidates and returns one with the largest score among those candidates. In code, the stakeholder terms are calculated in `_evaluate_schedule`; compliance and penalties are calculated in `check_ethical_compliance`; and candidate generation and selection are performed by `optimize_schedule`. The fixed weights and penalties express the model's chosen priorities; they do not guarantee that all stakeholder interests are equally represented.

The ethical conditions are **soft constraints** in this scoring rule: a violation lowers the score but does not exclude a schedule from consideration. Consequently, a returned schedule may be noncompliant. The separate compliance report is necessary to distinguish a high-scoring schedule from one that satisfies every listed principle.

The procedure is a stochastic sampling search, not exhaustive enumeration. With $n$ employees, $T$ periods, and $K$ candidates, candidate construction and schedule evaluation take approximately $O(KnT)$ time; each candidate schedule uses $O(nT)$ space. Repeated runs can select different schedules unless the random-number generator is reset to the same seed before the search.

In [ ]:
from dataclasses import dataclass
from typing import Callable, Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

## 2. Representing stakeholders, constraints, and metrics

A stakeholder stores a utility function and its weight in the combined score. An ethical constraint stores a Boolean test and the penalty applied when the test fails. A value metric stores a function and optional target information for reporting or interpretation.

Separating these concepts makes the model's components explicit: utilities contribute to the score, ethical tests determine compliance and penalties, and value metrics provide measurements. A metric's target and tolerance are metadata here; they do not affect candidate generation or the score.

In [ ]:
@dataclass
class Stakeholder:
    """Store a stakeholder's schedule utility and its score weight."""

    name: str
    utility_function: Callable[[np.ndarray], float]
    weight: float
    constraints: List[str]


@dataclass
class EthicalConstraint:
    """Store a Boolean ethical test and its violation penalty."""

    name: str
    constraint_function: Callable[[np.ndarray], bool]
    penalty_weight: float
    description: str


@dataclass
class ValueMetric:
    """Store a schedule metric and optional interpretation targets."""

    name: str
    metric_function: Callable[[np.ndarray], float]
    target_value: float
    tolerance: float

## 3. Scheduler implementation

The scheduler keeps the framework's main steps visible: register stakeholders and principles, calculate utility and compliance, generate a candidate, score it, and retain the best candidate encountered. Candidate schedules are generated period by period by sampling distinct employees, which satisfies the demand equality whenever demand does not exceed the available workforce.

In [ ]:
class ValueAlignedScheduler:
    """Score sampled workforce schedules using utilities and penalties."""

    def __init__(self):
        self.stakeholders = []
        self.ethical_constraints = []
        self.value_metrics = []
        self.schedule_history = []

    def add_stakeholder(self, stakeholder: Stakeholder) -> None:
        """Register a stakeholder whose utility contributes to the score."""
        self.stakeholders.append(stakeholder)

    def add_ethical_constraint(self, constraint: EthicalConstraint) -> None:
        """Register a compliance test and its violation penalty."""
        self.ethical_constraints.append(constraint)

    def add_value_metric(self, metric: ValueMetric) -> None:
        """Register a metric for reporting on schedules."""
        self.value_metrics.append(metric)

    def calculate_stakeholder_utility(
        self, schedule: np.ndarray, stakeholder: Stakeholder
    ) -> float:
        """Evaluate one stakeholder's utility for a schedule."""
        return float(stakeholder.utility_function(schedule))

    def check_ethical_compliance(self, schedule: np.ndarray) -> Tuple[bool, float]:
        """Return overall compliance and the sum of violated penalties."""
        total_penalty = 0.0
        compliant = True

        for constraint in self.ethical_constraints:
            if not constraint.constraint_function(schedule):
                compliant = False
                total_penalty += constraint.penalty_weight

        return compliant, total_penalty

    def calculate_value_metrics(self, schedule: np.ndarray) -> Dict[str, float]:
        """Evaluate all registered reporting metrics for a schedule."""
        return {
            metric.name: float(metric.metric_function(schedule))
            for metric in self.value_metrics
        }

    def optimize_schedule(
        self, demand_schedule: np.ndarray, employee_pool: np.ndarray
    ) -> np.ndarray:
        """Return the highest-scoring schedule among 100 random candidates."""
        demand = np.asarray(demand_schedule)
        employees = np.asarray(employee_pool)

        if demand.ndim != 1 or employees.ndim != 1:
            raise ValueError("Demand and employee_pool must be one-dimensional.")
        if len(employees) == 0:
            raise ValueError("employee_pool must contain at least one employee.")
        if not np.all(np.isfinite(demand)):
            raise ValueError("Demand values must be finite.")
        if np.any(demand < 0) or np.any(demand != np.floor(demand)):
            raise ValueError("Demand values must be nonnegative integers.")
        if np.any(demand > len(employees)):
            raise ValueError(
                "Demand in each period must not exceed the employee-pool size."
            )

        best_schedule = None
        best_score = -np.inf

        # Comparing the same fixed number of candidates keeps the sampling
        # procedure explicit and makes its computational effort predictable.
        for _ in range(100):
            candidate = self._generate_candidate_schedule(demand, employees)
            score = self._evaluate_schedule(candidate)

            if score > best_score:
                best_score = score
                best_schedule = candidate.copy()

        return best_schedule

    def _generate_candidate_schedule(
        self, demand: np.ndarray, employees: np.ndarray
    ) -> np.ndarray:
        """Sample distinct employees for each period to meet its demand."""
        periods = len(demand)
        schedule = np.zeros((len(employees), periods), dtype=int)

        for period in range(periods):
            required = int(demand[period])
            available = len(employees)

            if required > 0:
                assigned = np.random.choice(available, required, replace=False)
                schedule[assigned, period] = 1

        return schedule

    def _evaluate_schedule(self, schedule: np.ndarray) -> float:
        """Combine weighted stakeholder utilities and ethical penalties."""
        if schedule is None:
            return -np.inf

        total_utility = 0.0
        for stakeholder in self.stakeholders:
            utility = self.calculate_stakeholder_utility(schedule, stakeholder)
            total_utility += stakeholder.weight * utility

        compliant, penalty = self.check_ethical_compliance(schedule)
        if not compliant:
            total_utility -= penalty

        return total_utility

## 4. Stakeholder utility functions

Each utility maps a schedule to a numerical preference score. These functions are deliberately simple proxies so their effect on the combined score can be inspected. They are not monetary values and should not be interpreted as complete measures of stakeholder well-being.

In [ ]:
def employee_utility(schedule: np.ndarray) -> float:
    """Reward similar workloads, using a normalized dispersion measure."""
    workloads = np.sum(schedule, axis=1)
    mean_workload = np.mean(workloads)

    if mean_workload == 0:
        return 0.0

    workload_balance = 1.0 - np.std(workloads) / mean_workload
    return float(workload_balance)


def customer_utility(schedule: np.ndarray) -> float:
    """Reward having at least one assigned employee in each period."""
    period_coverage = np.mean(np.sum(schedule, axis=0) > 0)
    return float(period_coverage)


def shareholder_utility(schedule: np.ndarray) -> float:
    """Use fewer total assignments as a simple cost-efficiency proxy."""
    total_staff = np.sum(schedule)
    return float(1.0 / (1.0 + total_staff))


def community_utility(schedule: np.ndarray) -> float:
    """Reward providing at least some work to more employees."""
    employment_stability = np.mean(np.sum(schedule, axis=1) > 0)
    return float(employment_stability)

## 5. Ethical tests

The fairness test compares the largest workload with the smallest **positive** workload. Employees with zero assigned hours are not included in that ratio; the separate equity test measures how many employees receive work. The sustainability test limits total assignments. All three tests are Boolean checks whose failures contribute penalties in the scoring function.

In [ ]:
def fairness_constraint(schedule: np.ndarray) -> bool:
    """Require the largest positive workload to be at most twice the smallest."""
    workloads = np.sum(schedule, axis=1)
    positive_workloads = workloads[workloads > 0]

    if len(positive_workloads) == 0:
        return True

    max_workload = np.max(positive_workloads)
    min_workload = np.min(positive_workloads)
    return bool(max_workload <= 2 * min_workload)


def sustainability_constraint(schedule: np.ndarray) -> bool:
    """Limit total assignments under the model's staffing-count proxy."""
    total_staff = np.sum(schedule)
    return bool(total_staff <= 50)


def equity_constraint(schedule: np.ndarray) -> bool:
    """Require at least 80 percent of employees to receive some work."""
    workloads = np.sum(schedule, axis=1)
    employed_count = np.sum(workloads > 0)
    total_employees = len(workloads)
    return bool(employed_count >= 0.8 * total_employees)

## 6. Configure the example

The demand profile represents a 24-period day with higher demand around the peak. The random seed makes the sampling procedure reproducible when the notebook is run from this point in a fresh kernel. Since each generated candidate meets demand exactly, the total number of assignments in any candidate equals the sum of the period demands. Here that required total exceeds the sustainability limit of 50, so no candidate can satisfy all three ethical tests. The sustainability test remains useful for demonstrating how a soft penalty and a separate compliance report behave when a stated principle conflicts with required coverage.

In [ ]:
scheduler = ValueAlignedScheduler()

scheduler.add_stakeholder(
    Stakeholder("Employees", employee_utility, 0.3, [])
)
scheduler.add_stakeholder(
    Stakeholder("Customers", customer_utility, 0.3, [])
)
scheduler.add_stakeholder(
    Stakeholder("Shareholders", shareholder_utility, 0.2, [])
)
scheduler.add_stakeholder(
    Stakeholder("Community", community_utility, 0.2, [])
)

scheduler.add_ethical_constraint(
    EthicalConstraint(
        "Fairness", fairness_constraint, 10.0, "Fair workload distribution"
    )
)
scheduler.add_ethical_constraint(
    EthicalConstraint(
        "Sustainability",
        sustainability_constraint,
        5.0,
        "Environmental sustainability proxy",
    )
)
scheduler.add_ethical_constraint(
    EthicalConstraint(
        "Equity", equity_constraint, 8.0, "Equitable work opportunities"
    )
)

np.random.seed(42)
periods = 24
employees = 20

demand_schedule = np.array(
    [
        2, 2, 2, 2, 3, 4, 6, 8, 10, 12, 14, 15,
        15, 14, 12, 10, 8, 6, 4, 3, 2, 2, 2, 2,
    ]
)
employee_pool = np.arange(employees)

assert len(demand_schedule) == periods
assert np.all(demand_schedule <= len(employee_pool))

print("Value-aligned scheduler initialized with:")
print(f"- {len(scheduler.stakeholders)} stakeholders")
print(f"- {len(scheduler.ethical_constraints)} ethical tests")
print(f"- {employees} employees for {periods} hourly periods")
print(f"- Total required assignments: {int(np.sum(demand_schedule))}")

## 7. Generate and evaluate candidate schedules

The search draws 100 candidate schedules and retains the one with the highest weighted utility after applicable penalties. The checks below verify the binary assignment structure and confirm that every period's staffing equals its demand. The compliance result is reported separately because the score can favor a schedule even when a soft ethical test fails.

In [ ]:
optimal_schedule = scheduler.optimize_schedule(demand_schedule, employee_pool)

assert optimal_schedule.shape == (len(employee_pool), len(demand_schedule))
assert np.all((optimal_schedule == 0) | (optimal_schedule == 1))
assert np.array_equal(
    np.sum(optimal_schedule, axis=0), demand_schedule
)

print("=== Value-aligned schedule evaluation ===")
print(f"Schedule shape: {optimal_schedule.shape}")
print(f"Total staff assignments: {int(np.sum(optimal_schedule))}")
print(
    "Average staff per period: "
    f"{np.mean(np.sum(optimal_schedule, axis=0)):.2f}"
)

compliant, penalty = scheduler.check_ethical_compliance(optimal_schedule)
print(f"\nEthical compliance: {'PASS' if compliant else 'FAIL'}")
if not compliant:
    print(f"Total penalty applied in the score: {penalty:.2f}")

print("\n=== Stakeholder utilities ===")
for stakeholder in scheduler.stakeholders:
    utility = scheduler.calculate_stakeholder_utility(
        optimal_schedule, stakeholder
    )
    print(f"{stakeholder.name}: {utility:.3f}")

## 8. Visualize coverage, utilities, and workload distribution

The heatmap shows which employees are assigned in each period. The coverage plot compares required demand with assigned staffing; the two lines should coincide because candidate generation meets demand exactly. Utility scores are stakeholder-specific proxies on different conceptual dimensions, so their values are best read alongside their definitions rather than as directly comparable measures of welfare. The workload plot and printed diagnostics describe how assignments are distributed across employees.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle(
    "Value-Aligned Workforce Scheduling",
    fontsize=16,
    fontweight="bold",
)

ax1 = axes[0, 0]
sns.heatmap(
    optimal_schedule,
    cmap="Blues",
    ax=ax1,
    cbar_kws={"label": "Assignment (0 = no, 1 = yes)"},
)
ax1.set_title("Employee Assignments by Period")
ax1.set_xlabel("Time Period (hour)")
ax1.set_ylabel("Employee Index")

ax2 = axes[0, 1]
period_indices = np.arange(len(demand_schedule))
staff_per_period = np.sum(optimal_schedule, axis=0)
ax2.plot(
    period_indices,
    demand_schedule,
    color="red",
    linewidth=2,
    marker="o",
    label="Demand",
)
ax2.plot(
    period_indices,
    staff_per_period,
    color="blue",
    linewidth=2,
    marker="s",
    label="Staff assigned",
)
ax2.fill_between(
    period_indices,
    demand_schedule,
    staff_per_period,
    where=staff_per_period < demand_schedule,
    interpolate=True,
    alpha=0.3,
    color="red",
    label="Staffing shortfall",
)
ax2.fill_between(
    period_indices,
    demand_schedule,
    staff_per_period,
    where=staff_per_period > demand_schedule,
    interpolate=True,
    alpha=0.3,
    color="green",
    label="Staffing surplus",
)
ax2.set_title("Demand and Assigned Staffing")
ax2.set_xlabel("Time Period (hour)")
ax2.set_ylabel("Number of employees")
ax2.legend()
ax2.grid(True, alpha=0.3)

ax3 = axes[1, 0]
stakeholder_names = [stakeholder.name for stakeholder in scheduler.stakeholders]
utilities = [
    scheduler.calculate_stakeholder_utility(optimal_schedule, stakeholder)
    for stakeholder in scheduler.stakeholders
]
colors = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#FFA07A"]
bars = ax3.bar(stakeholder_names, utilities, color=colors)
ax3.set_title("Stakeholder Utility Scores")
ax3.set_ylabel("Utility score")
ax3.tick_params(axis="x", rotation=15)
for bar, utility in zip(bars, utilities):
    ax3.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{utility:.3f}",
        ha="center",
        va="bottom",
    )

ax4 = axes[1, 1]
workloads = np.sum(optimal_schedule, axis=1)
employee_indices = np.arange(len(workloads))
ax4.bar(employee_indices, workloads, color="skyblue", alpha=0.7)
mean_workload = np.mean(workloads)
std_workload = np.std(workloads)
ax4.axhline(
    y=mean_workload,
    color="red",
    linestyle="--",
    label=f"Mean: {mean_workload:.1f} hours",
)
ax4.axhline(
    y=mean_workload + std_workload,
    color="orange",
    linestyle="--",
    label=f"Mean + 1 SD: {mean_workload + std_workload:.1f} hours",
)
ax4.axhline(
    y=mean_workload - std_workload,
    color="goldenrod",
    linestyle=":",
    label=f"Mean - 1 SD: {mean_workload - std_workload:.1f} hours",
)
ax4.set_title("Workload Distribution")
ax4.set_xlabel("Employee Index")
ax4.set_ylabel("Assigned work hours")
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

workload_mean = np.mean(workloads)
workload_std = np.std(workloads)
coefficient_of_variation = (
    workload_std / workload_mean if workload_mean != 0 else np.nan
)
employees_with_work = np.sum(workloads > 0)

print("\n=== Fairness and equity diagnostics ===")
print(f"Workload standard deviation: {workload_std:.2f} hours")
print(f"Workload coefficient of variation: {coefficient_of_variation:.3f}")
print(
    f"Employees with work: {employees_with_work}/{len(workloads)} "
    f"({100 * employees_with_work / len(workloads):.1f}%)"
)
print(f"Fairness test satisfied: {fairness_constraint(optimal_schedule)}")
print(
    "Sustainability test satisfied: "
    f"{sustainability_constraint(optimal_schedule)}"
)
print(f"Equity test satisfied: {equity_constraint(optimal_schedule)}")

print("\n=== Interpretation ===")
print(
    "The schedule meets period demand by construction. "
    "Ethical compliance is assessed independently of the score: "
    "a penalty can reduce the score, but it does not make a violating "
    "candidate ineligible. The sustainability threshold is a staffing "
    "proxy and should not be treated as a measured environmental impact."
)